# CEM vs PCM Validation — 2035 GTEP Solution

**Purpose:** Quantify the modeling gap between the Capacity Expansion Model (CEM = GTEP)
and the Production Cost Model (PCM = Prescient) for the 2035 investment stage.

**Factorial design:** 2 EP scenarios × 2 PCM configs = 4 runs

| | Config A (PTDF, sh=6, rh=36) | Config B (btheta, sh=24, rh=48) |
|---|---|---|
| **No-extreme** | no_extreme_A | no_extreme_B |
| **Extreme** | extreme_A | extreme_B |

**Key question:** How much does the GTEP's aggregated cost estimate
(4 representative days × linear cost model) differ from Prescient's
365-day chronological simulation with piecewise HR curves?

---

> **Note:** Extreme scenario results are missing `overall_simulation_output.csv`.
> Those cells will be flagged as INVALID but still load available data.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import warnings

warnings.filterwarnings('ignore', category=FutureWarning)
plt.rcParams.update({'figure.dpi': 120, 'figure.figsize': (12, 5)})

# Ensure repo root is on path for imports
# notebooks/ is 4 levels deep: idaes-gtep / gtep / pcm_analysis / cem_vs_pcm_validation_2035 / notebooks
REPO_ROOT = Path.cwd().resolve().parents[3]
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from gtep.pcm_analysis.cem_vs_pcm_validation_2035.src.validation_utils import (
    run_registry, load_run, headline_metrics, factorial_dataframe,
    load_weighted_lmp, load_weighted_lmp_hourly,
    marginal_cost_sanity, load_gtep_stage3_cost,
    compute_tsa_error, rep_day_attribution, banner_title,
    EXPECTED_FUEL_MC, GTEP_REP_WEIGHT,
)

print(f'Repo root: {REPO_ROOT}')
print(f'GTEP rep weight: {GTEP_REP_WEIGHT}')

## 1. Load All Runs

In [ ]:
registry = run_registry()
datasets = {}

for key, spec in registry.items():
    print(f'Loading {key} ...')
    print(f'  results_dir: {spec.results_dir}')
    print(f'  exists: {spec.results_dir.exists()}')
    data = load_run(spec, heavy=True)
    datasets[key] = data
    status = 'VALID' if data.valid else f'INVALID: {data.reasons}'
    print(f'  status: {status}')
    if data.overall is not None:
        o = data.overall.iloc[0]
        print(f'  total_cost: ${o.get("Total costs", float("nan")):,.0f}')
    print()

## 2. Factorial Summary Table

In [ ]:
factorial_df = factorial_dataframe(datasets)

display_cols = [
    'key', 'valid', 'scenario', 'config',
    'total_demand_TWh', 'total_gen_cost_B$', 'total_cost_B$',
    'cumulative_avg_price_$MWh', 'load_shedding_MWh',
    'curtailment_MWh_overall', 'reserve_shortfall_MWh',
]
available_cols = [c for c in display_cols if c in factorial_df.columns]
print('Factorial Summary')
print('=' * 100)
display(factorial_df[available_cols].round(4))

# Save
out_path = REPO_ROOT / 'gtep' / 'pcm_analysis' / 'cem_vs_pcm_validation_2035' / 'results' / 'factorial_summary.csv'
factorial_df.to_csv(out_path, index=False)
print(f'\nSaved to {out_path}')

## 3. Marginal Cost Validation

Verify that observed dispatch costs match expected fuel_cost3 values:
- NUC: $7.38/MWh
- COAL: $18.94/MWh  
- CT: $22.80/MWh

In [ ]:
for key, data in datasets.items():
    if not data.valid:
        print(f'\n{key}: SKIPPED (invalid){data.banner}')
        continue
    mc = marginal_cost_sanity(data)
    if mc.empty:
        print(f'\n{key}: no dispatch data')
        continue
    print(f'\n{key}:')
    print(mc.to_string(index=False))
    
    for _, row in mc.iterrows():
        utype = row['Unit Type']
        pct = row['pct_err']
        status = 'PASS' if pct < 20 else 'FAIL'
        print(f'  {status} {utype}: {pct:.1f}% error')

## 4. Load-Weighted LMP Comparison

In [ ]:
lmp_results = {}
for key, data in datasets.items():
    if data.bus_detail is None:
        print(f'{key}: no bus_detail — skipping LMP')
        continue
    lw = load_weighted_lmp(data.bus_detail)
    lmp_results[key] = lw
    print(f'{key}: LW-LMP = ${lw:.2f}/MWh{data.banner}')

if lmp_results:
    print(f'\nSpread: ${max(lmp_results.values()) - min(lmp_results.values()):.2f}/MWh')

In [ ]:
# Hourly LMP comparison for valid runs
fig, axes = plt.subplots(2, 1, figsize=(14, 8), sharex=True)

colors = {'no_extreme_A': '#1f77b4', 'no_extreme_B': '#ff7f0e',
          'extreme_A': '#2ca02c', 'extreme_B': '#d62728'}

for ax_idx, scenario in enumerate(['no_extreme', 'extreme']):
    ax = axes[ax_idx]
    for key, data in datasets.items():
        if data.spec.scenario != scenario or data.bus_detail is None:
            continue
        hourly = load_weighted_lmp_hourly(data.bus_detail)
        ax.plot(hourly['Datetime'], hourly['LW_LMP'],
                linewidth=0.3, alpha=0.7, color=colors[key],
                label=f'{data.spec.config}{data.banner}')
    ax.set_ylabel('$/MWh')
    ax.set_title(f'{scenario.replace("_", "-")} scenario')
    ax.legend(loc='upper right')

plt.suptitle('Hourly Load-Weighted LMP by Scenario and Config', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 5. Generation Mix Comparison

In [ ]:
fuel_order = ['NUC', 'COAL', 'CT', 'HYDRO', 'WIND', 'PV']
mix_rows = []

for key, data in datasets.items():
    if data.thermal_detail is None or data.gen is None:
        continue
    gen_info = data.gen[['GEN UID', 'Unit Type']].rename(columns={'GEN UID': 'Generator'})
    
    # Thermal
    therm = data.thermal_detail.merge(gen_info, on='Generator', how='left')
    therm_gwh = therm.groupby('Unit Type')['Dispatch'].sum() / 1e3
    
    # Renewable
    renew_info = data.gen[data.gen['Unit Type'].isin(['PV', 'WIND', 'HYDRO'])][['GEN UID', 'Unit Type']]
    renew_info = renew_info.rename(columns={'GEN UID': 'Generator'})
    ren = data.renewables_detail.merge(renew_info, on='Generator', how='left')
    renew_gwh = ren.groupby('Unit Type')['Output'].sum() / 1e3
    
    combined = pd.concat([therm_gwh, renew_gwh])
    row = {'key': key, 'scenario': data.spec.scenario, 'config': data.spec.config}
    for ft in fuel_order:
        row[f'{ft}_GWh'] = combined.get(ft, 0)
    row['total_GWh'] = sum(row[f'{ft}_GWh'] for ft in fuel_order)
    row['banner'] = data.banner
    mix_rows.append(row)

mix_df = pd.DataFrame(mix_rows)
print('Generation Mix by Run (GWh)')
print('=' * 100)
display(mix_df.round(1))

In [ ]:
# Stacked bar chart
if not mix_df.empty:
    COLORS = {'NUC': '#e41a1c', 'COAL': '#555555', 'CT': '#ff7f00',
              'WIND': '#4daf4a', 'PV': '#ffff33', 'HYDRO': '#377eb8'}
    
    fig, ax = plt.subplots(figsize=(10, 6))
    x = range(len(mix_df))
    bottom = np.zeros(len(mix_df))
    
    for ft in fuel_order:
        col = f'{ft}_GWh'
        if col in mix_df.columns:
            vals = mix_df[col].values
            ax.bar(x, vals, bottom=bottom, label=ft,
                   color=COLORS.get(ft, '#999999'), alpha=0.85)
            bottom += vals
    
    ax.set_xticks(x)
    ax.set_xticklabels(mix_df['key'], rotation=30, ha='right')
    ax.set_ylabel('GWh')
    ax.set_title('Annual Generation by Fuel Type — Factorial Comparison')
    ax.legend(loc='upper right')
    plt.tight_layout()
    plt.show()

## 6. Config Sensitivity: PTDF vs btheta

For each scenario, compute the delta between Config A and Config B.

In [ ]:
for scenario in ['no_extreme', 'extreme']:
    key_a = f'{scenario}_A'
    key_b = f'{scenario}_B'
    da, db = datasets.get(key_a), datasets.get(key_b)
    
    if da is None or db is None:
        print(f'{scenario}: missing run(s)')
        continue
    if da.overall is None or db.overall is None:
        print(f'{scenario}: missing overall stats{da.banner}{db.banner}')
        continue
    
    oa, ob = da.overall.iloc[0], db.overall.iloc[0]
    
    print(f'\n{scenario.upper()} — Config A vs Config B')
    print('=' * 70)
    for col in ['Total costs', 'Total generation costs', 'Total fixed costs',
                'Cumulative average price', 'Total load shedding',
                'Total renewables curtailment', 'Total reserve shortfall']:
        va = float(oa.get(col, float('nan')))
        vb = float(ob.get(col, float('nan')))
        delta = vb - va
        pct = delta / va * 100 if va != 0 else float('nan')
        print(f'  {col:<35s}  A={va:>14,.2f}  B={vb:>14,.2f}  Δ={delta:>+12,.2f} ({pct:>+.2f}%)')

## 7. CEM vs PCM Gap (GTEP Operating Cost vs Prescient Total Gen Cost)

The GTEP CEM estimates annual operating cost using 4 representative days × weights.
Prescient PCM simulates all 365 days with full chronological detail.

> **Requires:** `gtep_stage3_cost.json` from running `extract_gtep_stage3_cost.py` on CRC.
> If the JSON is missing, this section shows placeholder values.

In [ ]:
gtep_cost = load_gtep_stage3_cost()

if gtep_cost is None:
    print('WARNING: gtep_stage3_cost.json not found.')
    print('Run on CRC: python gtep/pcm_analysis/cem_vs_pcm_validation_2035/src/extract_gtep_stage3_cost.py')
    print('\nShowing PCM costs only (GTEP side will be NaN).')
else:
    print('GTEP cost extraction loaded.')
    print(f'  Scenario: {gtep_cost.get("scenario", "?")}')
    print(f'  Stage: {gtep_cost.get("stage_index", "?")}')
    print(f'  5yr op cost: ${gtep_cost.get("total_operating_cost_stage3_5yr", float("nan")):,.0f}')
    print(f'  1yr op cost: ${gtep_cost.get("total_operating_cost_stage3_1yr", float("nan")):,.0f}')
    print(f'  Objective:   ${gtep_cost.get("objective_value", float("nan")):,.0f}')
    print(f'  Rep dates:   {gtep_cost.get("representative_dates", [])}')

# Compute gap for each valid run
print('\n' + '=' * 80)
print('CEM vs PCM Gap')
print('=' * 80)

gap_rows = []
for key, data in datasets.items():
    if data.overall is None:
        print(f'{key}: no overall data{data.banner}')
        continue
    pcm_gen_cost = float(data.overall.iloc[0].get('Total generation costs', float('nan')))
    gap = compute_tsa_error(gtep_cost, pcm_gen_cost)
    gap['key'] = key
    gap['scenario'] = data.spec.scenario
    gap['config'] = data.spec.config
    gap_rows.append(gap)
    print(f'\n{key}:')
    for k, v in gap.items():
        if k in ('key', 'scenario', 'config'):
            continue
        if isinstance(v, float):
            print(f'  {k}: {v:,.4f}')
        else:
            print(f'  {k}: {v}')

gap_df = pd.DataFrame(gap_rows)
gap_path = REPO_ROOT / 'gtep' / 'pcm_analysis' / 'cem_vs_pcm_validation_2035' / 'results' / 'cem_vs_pcm_gap.csv'
gap_df.to_csv(gap_path, index=False)
print(f'\nSaved to {gap_path}')

## 8. Representative Day Attribution

Decompose the CEM-PCM gap into per-representative-day contributions:
- **Sampling error:** GTEP's cost on that day vs PCM's cost on the same calendar day
- **Aggregation error:** PCM day cost × weight vs PCM full-quarter cost

> Requires GTEP cost JSON.

In [ ]:
if gtep_cost is not None:
    for key in ['no_extreme_A', 'no_extreme_B']:
        data = datasets.get(key)
        if data is None or data.overall is None:
            continue
        attr = rep_day_attribution(data, gtep_cost)
        if attr.empty:
            print(f'{key}: no attribution data')
            continue
        print(f'\n{key} — Rep Day Attribution')
        print('=' * 120)
        display(attr.round(2))
        
        # Save
        attr_path = REPO_ROOT / 'gtep' / 'pcm_analysis' / 'cem_vs_pcm_validation_2035' / 'results' / f'rep_day_attribution_{key}.csv'
        attr.to_csv(attr_path, index=False)
        print(f'Saved to {attr_path}')
else:
    print('Skipped — GTEP cost JSON not available.')
    print('Run: python gtep/pcm_analysis/cem_vs_pcm_validation_2035/src/extract_gtep_stage3_cost.py')

## 9. Quarterly Cost Comparison

In [ ]:
for key, data in datasets.items():
    if data.thermal_detail is None:
        continue
    tmp = data.thermal_detail[['Datetime', 'Unit Cost']].copy()
    tmp['Quarter'] = tmp['Datetime'].dt.quarter
    q_cost = tmp.groupby('Quarter')['Unit Cost'].sum() / 1e9
    
    print(f'\n{key} — Quarterly Generation Cost ($B){data.banner}')
    for q, v in q_cost.items():
        print(f'  Q{q}: ${v:.4f}B')
    print(f'  Total: ${q_cost.sum():.4f}B')

## 10. Summary & Key Findings

In [ ]:
print('CEM vs PCM Validation — Key Findings')
print('=' * 70)

# Valid runs summary
valid_runs = {k: d for k, d in datasets.items() if d.valid}
invalid_runs = {k: d for k, d in datasets.items() if not d.valid}
print(f'Valid runs: {len(valid_runs)}/{len(datasets)}')
if invalid_runs:
    print(f'Invalid runs: {", ".join(invalid_runs.keys())}')

# LMP spread
if len(lmp_results) >= 2:
    vals = list(lmp_results.values())
    print(f'\nLW-LMP range: ${min(vals):.2f} – ${max(vals):.2f}/MWh (spread: ${max(vals)-min(vals):.2f})')

# Config sensitivity
for scenario in ['no_extreme']:
    ka, kb = f'{scenario}_A', f'{scenario}_B'
    if ka in lmp_results and kb in lmp_results:
        delta = lmp_results[kb] - lmp_results[ka]
        print(f'\n{scenario} config sensitivity (B - A): ${delta:+.2f}/MWh')

# CEM-PCM gap
if gtep_cost is not None and not gap_df.empty:
    for _, row in gap_df.iterrows():
        if row.get('tsa_error_pct') == row.get('tsa_error_pct'):  # not NaN
            print(f'\nCEM-PCM gap ({row["key"]}): {row["tsa_error_pct"]:+.2f}%')
else:
    print('\nCEM-PCM gap: awaiting GTEP cost extraction (run on CRC)')

print('\n' + '=' * 70)
print('Done.')